# Ropedia Academy — B9 · Paper deep-dive & reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B9.ipynb)

> **A complete miniature NeRF training run on toy posed rays — representation, differentiable renderer, photometric loss, plus a density-sparsity term that suppresses floaters.**
>
> 在玩具位姿射线上完整跑一次微型 NeRF 训练——表示、可微渲染器、光度损失，外加抑制漂浮物的密度稀疏项。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B9

In [ ]:
import torch, torch.nn as nn

# Minimal end-to-end NeRF reproduction: representation -> renderer -> photometric loss.
def posenc(x, L=4):
    return torch.cat([x] + [f(x*(2.0**i)) for i in range(L) for f in (torch.sin, torch.cos)], -1)
mlp = nn.Sequential(nn.Linear(3*(1+2*4), 128), nn.ReLU(), nn.Linear(128, 4))
def field(p):
    o = mlp(posenc(p)); return torch.sigmoid(o[..., :3]), torch.relu(o[..., 3])
def render(p, delta):
    c, s = field(p); a = 1 - torch.exp(-s * delta)
    T = torch.cumprod(torch.cat([torch.ones(p.shape[0],1), 1-a+1e-10], 1)[:, :-1], 1)
    return (T * a).unsqueeze(-1).mul(c).sum(1), a

R, S = 8, 32                                          # toy "posed rays" dataset
t = torch.linspace(2, 6, S); delta = (t[1]-t[0]).expand(R, S)
pts = torch.randn(R,1,3)*0.1 + torch.stack([torch.zeros(S), torch.zeros(S), t], -1)
gt  = torch.rand(R, 3)
opt = torch.optim.Adam(mlp.parameters(), 1e-3)
for _ in range(200):
    opt.zero_grad(); pix, a = render(pts, delta)
    photo = ((pix - gt)**2).mean()
    (photo + 1e-3 * a.mean()).backward(); opt.step()  # sparsity term suppresses FLOATERS
print("photometric loss:", round(photo.item(), 4), "| mean density (floater control):", round(a.mean().item(), 3))

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B9
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks